# RFM Customer Segmentation
Rule-based and KMeans customer segmentation using the real Olist PostgreSQL database.

## 1. Business Problem
Identify actionable customer groups for onboarding, loyalty, retention, and reactivation while recognizing that only about 3% of eligible customers repeat.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT_ROOT))
from src.models.customer_segmentation.build_rfm import build_rfm_dataset
from src.models.customer_segmentation.preprocess import add_rfm_scores
from src.models.customer_segmentation.train import main as train_segmentation
from src.models.customer_segmentation.predict import predict_segment

INFO: generated new fontManager


## 2. Customer Dataset
The PostgreSQL extract excludes canceled and unavailable orders, aggregates payments to order grain, and produces exactly one row per `customer_unique_id`.

In [2]:
rfm=add_rfm_scores(build_rfm_dataset())
print('Shape:',rfm.shape,'duplicate customers:',rfm.customer_unique_id.duplicated().sum())
display(rfm.head())

Shape: (94990, 13) duplicate customers: 0


,customer_unique_id,first_purchase,last_purchase,recency,frequency,monetary,customer_state,customer_city,reference_date,r_score,m_score,f_score,rfm_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,117,1,141.90,SP,cajamar,2018-09-04,4,4,1,Promising
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,120,1,27.19,SP,osasco,2018-09-04,4,1,1,Promising
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,543,1,86.22,SC,sao jose,2018-09-04,1,2,1,Hibernating
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,327,1,43.62,PA,belem,2018-09-04,2,1,1,Hibernating
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,294,1,196.89,SP,sorocaba,2018-09-04,2,4,1,High Value At Risk


## 3. RFM Definition
Recency is days since the final eligible purchase at a fixed reference date; Frequency is distinct eligible orders; Monetary is total order-level payment value.

In [3]:
print('Reference date:',rfm.reference_date.iloc[0].date())
print('Monetary total:',rfm.monetary.sum())

Reference date: 2018-09-04
Monetary total: 15739137.010000002


## 4. RFM Data Quality
Negative values and duplicate business keys fail validation. Extreme customers remain included and are described rather than silently deleted.

In [4]:
display(rfm[['recency','frequency','monetary']].describe(percentiles=[.5,.9,.95,.99]).T)
display(rfm[['recency','frequency','monetary']].skew().rename('skewness'))

,count,mean,std,min,50%,90%,95%,99%,max
recency,94990.0,244.349479,153.000542,1.0,225.0,473.00,526.00,581.0000,730.00
frequency,94990.0,1.033867,0.210826,1.0,1.0,1.00,1.00,2.0000,16.00
monetary,94990.0,165.692568,226.740288,0.0,107.9,318.97,472.94,1098.6618,13664.08


recency       0.447455
frequency    11.506949
monetary      9.107369
Name: skewness, dtype: float64

## 5. RFM Distributions
Frequency and Monetary are strongly skewed. Portfolio figures use labeled `log1p` axes while retaining every customer.

## 6. Rule-Based Segmentation
Recency and Monetary use rank-based quintiles. Frequency uses explicit 1, 2, 3, 4, and 5+ order bands because naive quantiles collapse when almost everyone has one order.

In [5]:
display(rfm.rfm_segment.value_counts().to_frame('customers'))

,customers
rfm_segment,
Hibernating,22975
Needs Attention,18962
Promising,18399
New Customers,18355
High Value At Risk,14769
Potential Loyalists,1114
At Risk,252
Champions,123
Loyal Customers,41


## 7. Feature Transformation
KMeans receives `log1p(Recency, Frequency, Monetary)` followed by train-wide standardization. Customer identifiers are never clustering features.

## 8. Selecting Number of Clusters
k=2 through 8 are compared by inertia, sampled silhouette, minimum cluster size, and usefulness. Selection is not based on a single metric.

## 9. KMeans Clustering
The following cell executes evaluation, selection, profiling, persistence, report generation, and analytical export.

In [6]:
train_segmentation()

INFO: Customers=94,990 reference=2018-09-04 selected_k=2 silhouette=0.7081


 cluster_id  customer_count  median_recency  median_frequency  median_monetary  mean_monetary  total_revenue  repeat_customer_rate  customer_percentage  revenue_share top_customer_states                cluster_name
          0           92102           226.0               1.0           105.65     161.219115    14848602.89                   0.0             96.95968      94.341913          SP, RJ, MG          One-Time Customers
          1            2888           206.5               2.0           225.53     308.356690      890534.12                   1.0              3.04032       5.658087          SP, RJ, MG Repeat High-Value Customers


## 10. Cluster Validation
Silhouette is estimated on a reproducible 10,000-customer sample to avoid quadratic full-dataset cost. Tiny clusters are explicitly checked.

In [7]:
report_dir=PROJECT_ROOT/'reports'/'segmentation'
display(pd.read_csv(report_dir/'k_selection_metrics.csv'))

,k,inertia,silhouette_score,minimum_cluster_size,minimum_cluster_share
0,2,193525.915081,0.708138,2888,0.030403
1,3,130828.612299,0.399328,2888,0.030403
2,4,84592.385951,0.391546,2888,0.030403
3,5,69872.446984,0.358921,2888,0.030403
4,6,59704.160790,0.356392,2888,0.030403
5,7,52106.691809,0.354414,2888,0.030403
6,8,46262.436333,0.350510,2888,0.030403


## 11. Customer Profiles
Names are assigned only after comparing customer share, RFM medians, revenue, repeat rate, and states.

In [8]:
profiles=pd.read_csv(report_dir/'cluster_profiles.csv')
display(profiles)

,cluster_id,customer_count,median_recency,median_frequency,median_monetary,mean_monetary,total_revenue,repeat_customer_rate,customer_percentage,revenue_share,top_customer_states,cluster_name
0,0,92102,226.0,1.0,105.65,161.219115,14848602.89,0.0,96.95968,94.341913,"SP, RJ, MG",One-Time Customers
1,1,2888,206.5,2.0,225.53,308.356690,890534.12,1.0,3.04032,5.658087,"SP, RJ, MG",Repeat High-Value Customers


## 12. PCA Visualization
PCA provides a two-dimensional view of transformed RFM space. Explained variance is reported, but the projection does not prove cluster validity.

## 13. Rule-Based vs ML Segmentation
The cross-tab below shows how interpretable business rules overlap with data-driven clusters.

In [9]:
display(pd.read_csv(report_dir/'rule_vs_cluster.csv',index_col=0))

,One-Time Customers,Repeat High-Value Customers
rfm_segment,,
At Risk,0.000000,1.000000
Champions,0.000000,1.000000
Hibernating,1.000000,0.000000
High Value At Risk,0.946916,0.053084
Loyal Customers,0.000000,1.000000
Needs Attention,0.969729,0.030271
New Customers,1.000000,0.000000
Potential Loyalists,0.000000,1.000000
Promising,1.000000,0.000000


## 14. Business Recommendations
Use segment profiles to design measurable onboarding, loyalty, and reactivation experiments rather than treating labels as permanent customer identities.

In [10]:
display(predict_segment(rfm[['recency','frequency','monetary']].tail(5)))

,cluster_id,cluster_name
94985,0,One-Time Customers
94986,0,One-Time Customers
94987,0,One-Time Customers
94988,0,One-Time Customers
94989,0,One-Time Customers


## 15. Limitations
RFM omits margin, acquisition, browsing, campaign exposure, and customer opportunity windows. Cluster stability should be tested across rolling snapshots, and business value requires controlled experiments.